# Vector GBT equivalence to Linear Regression

This short notebooks illustrates how a vector-valued GBT leaf estimation problem is equivalent to a linear regression problem, and how the sum of squared residuals in the linear regression formulation can be otained from the values of the shorter equivalent quadratic problem.

For this, it tries to estimate the optimal leaf values of a multinomial logistic objective if the first 5,000 observations of the CovType dataset were assigned to the same terminal node and had a previously existing base score, which here will be generated as random numbers.

Since the problem is unregularized, it sets the last class as the reference class, esimating 6 parameters in total.

# A look at the dataset

In [1]:
import numpy as np
from sklearn.datasets import fetch_covtype
_, y = fetch_covtype(return_X_y=True)
y = y[:5_000]

y = (y - 1).astype(int)
k = y.max() + 1
n = y.shape[0]

Y_wide = np.zeros((y.shape[0], y.max() + 1))
Y_wide[np.arange(n), y] = 1.
Y_wide.mean(axis=0)

array([0.1114, 0.1896, 0.1286, 0.2498, 0.189 , 0.0958, 0.0358])

# Multinomial logistic function, gradient and Hessian

These assume that the `X` supplied is a matrix of shape $n \times (k-1)$, with the current scores assigned to each observation. The aim is to find leaf values to add to these scores, such that the log-loss will be decreased the most, based on a second-order Newton approximation.

In [2]:
from scipy.special import softmax

# Function, gradient and Hessian for multinomial logistic
# with a reference class. Only 'k-1' raw scores are given.
def mnl_function(X, Y_wide):
    # last class is reference class, constrained to zero
    X = np.c_[X, np.zeros((X.shape[0], 1))]
    p = softmax(X, axis=1)
    return -np.sum(
        Y_wide*np.log(p) + (1-Y_wide)*np.log(1-p)
    )
def mnl_grad(X, Y_wide):
    k1 = X.shape[1]
    X = np.c_[X, np.zeros((X.shape[0], 1))]
    p = softmax(X, axis=1)
    return (Y_wide - p)[:, :k1]
def mnl_hess(X, Y_wide):
    k1 = X.shape[1]
    n = Y_wide.shape[0]
    k = Y_wide.shape[1]
    X = np.c_[X, np.zeros((n, 1))]
    p = softmax(X, axis=1)
    H = np.empty((n, k, k))
    for row in range(n):
        H[row] = np.diag(p[row]) - np.outer(p[row], p[row])
    return H[:, :k1, :k1]

# The GBT objective

This provides a reference solution, based on the most straightforward definition of the objective that GBTs are meant to minimize. Note that this is very inefficient and should not be done in practice - this only serves as a baseline.

In [3]:
from scipy.optimize import minimize

rng = np.random.default_rng(seed=123)
base_scores = rng.normal(size=(Y_wide.shape[0], Y_wide.shape[1]-1), scale=0.25)

def solve_long_form():
    G = mnl_grad(base_scores, Y_wide)
    H = mnl_hess(base_scores, Y_wide)

    B = np.vstack([
        np.hstack([
            np.diag(H[:, row, col])
            for col in range(k-1)
        ])
        for row in range(k-1)
    ])

    def fx_minimize(v):
        x_long = np.repeat(v, G.shape[0])
        return 0.5*(x_long@(B)@x_long) - x_long@G.reshape(-1, order="F")
    
    return minimize(fx_minimize, np.zeros(k-1), method="l-bfgs-b")
solve_long_form()["x"]

array([0.51693274, 1.07539921, 0.64304464, 1.48467234, 1.06328933,
       0.41110991])

# Linear Regression Equivalence

The GBT leaf estimation problem is equivalent to a linear regression problem, where in the 1D case, its residual sum of squares would be equivalent to the residual sum of squares from which the information gain is calculated. With more than 2 dimensions, the information gain formula is unclear, but the residual sum of squares is still well defined:

In [4]:
from sklearn.linear_model import LinearRegression

def solve_linreg_form():
    G = mnl_grad(base_scores, Y_wide)
    H = mnl_hess(base_scores, Y_wide)

    X_linreg_unshaped = np.zeros((n, k-1, k-1))
    Y_linreg_unshaped = np.zeros((X_linreg_unshaped.shape[0], k-1))
    for row in range(n):
        Chol_row = np.linalg.cholesky(H[row])
        X_linreg_unshaped[row] = Chol_row
        Y_linreg_unshaped[row] = np.linalg.solve(Chol_row, G[row])
    X_linreg = np.vstack([
        X_linreg_unshaped[:, :, cl]
        for cl in range(k-1)
    ])
    Y_linreg = Y_linreg_unshaped.reshape(-1, order="F")

    sol_linreg = LinearRegression(fit_intercept=False).fit(X_linreg, Y_linreg).coef_
    residuals = Y_linreg - X_linreg @ sol_linreg
    rss = residuals @ residuals
    return sol_linreg, rss

solve_linreg_form()

(array([0.51693282, 1.07539923, 0.64304471, 1.48467243, 1.06328945,
        0.41111002]),
 np.float64(30730.370570228813))

# Quadratic Program Equivalence

The GBT leaf estimation problem can be more easily put into a standard quadratic form to solve. The residual sum of squares can be recovered from the values of the function being minimized by this standard QP plus a constant that doesn't depend on the estimated variables.

In [ ]:
def solve_qp_form():
    G = mnl_grad(base_scores, Y_wide)
    H = mnl_hess(base_scores, Y_wide)

    A = H.sum(axis=0)
    b = G.sum(axis=0)
    sol = np.linalg.solve(A, b)

    qss = 0.5 * (sol @ A @ sol) - sol @ b
    rss_gap = 0.
    for row in range(n):
        rss_gap += G[row] @ np.linalg.inv(H[row]) @ G[row]
    return sol, 2*qss + rss_gap
solve_qp_form()

(array([0.51693282, 1.07539923, 0.64304471, 1.48467243, 1.06328945,
        0.41111002]),
 np.float64(30730.370570228744))